# Load Arancione Sales CSV files into Bronze — Idempotent MERGE

Reads all CSV files from the Arancione Volume folder and merges them into
`vinoworld.bronze.sales_arancione` using `row_hash` as the merge key.

Idempotent: re-running with the same files produces no duplicate rows.
Changed rows in a re-uploaded file are inserted as new hash versions;
original rows are preserved (Bronze is a versioned raw store).

Depends on: `/Workspace/Shared/notebook_init`, `/Workspace/Shared/catalog_setup`

In [0]:
%run "../../libs/notebook_init"

In [0]:
# =============================================================================
# Cell 2 — Constants
# All table references and paths derive from notebook_init constants.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime, timezone

STORE_NAME     = "Arancione"
SOURCE_SUBPATH = "arancione/"
SOURCE_PATH    = f"{RAW_FILES}{SOURCE_SUBPATH}"
TARGET_TABLE   = f"{BRONZE}.sales_arancione"

# Exact header order expected in every source CSV file.
EXPECTED_SOURCE_COLS = [
    "OnlineRetailer", "SalesMonth", "Title", "Vintage",
    "Variety", "Score", "ListPrice", "Quantity",
]

# Source business columns included in row_hash.
# Excludes all pipeline audit columns: inserted_ts, run_id, source_file_path, store_name.
HASH_SOURCE_COLS = [
    "online_retailer", "sales_month", "title", "vintage",
    "variety", "score", "list_price", "quantity",
]

# Explicit Bronze schema — all StringType. No casting at this layer.
read_schema = StructType([
    StructField("OnlineRetailer", StringType(), True),
    StructField("SalesMonth",     StringType(), True),
    StructField("Title",          StringType(), True),
    StructField("Vintage",        StringType(), True),
    StructField("Variety",        StringType(), True),
    StructField("Score",          StringType(), True),
    StructField("ListPrice",      StringType(), True),
    StructField("Quantity",       StringType(), True),
])

In [0]:
# =============================================================================
# Cell 3 — Step log init
# =============================================================================

nb = Utils.get_notebook_context(dbutils)
notebook_folder = nb["notebook_folder"]
notebook_name   = nb["notebook_name"]

step_log_id       = str(uuid.uuid4())
pipeline_run_id   = PIPELINE_RUN_ID
step_sequence     = 1
layer             = "bronze"
target_table      = TARGET_TABLE
status            = STATUS_RUNNING
started_timestamp = datetime.now(timezone.utc)
rows_read         = 0
rows_written      = 0
error_message     = None

pipeline_step_log_upsert(
    spark, step_log_id, pipeline_run_id, step_sequence,
    notebook_folder, notebook_name, status, started_timestamp,
    layer, target_table
)

In [0]:
# =============================================================================
# Cell 4 — File validation
# Probes each CSV file individually for header conformance before bulk read.
# Spark's directory read cannot detect per-file schema drift.
# =============================================================================

try:
    files = [
        f.path for f in dbutils.fs.ls(SOURCE_PATH)
        if f.path.lower().endswith(".csv")
    ]

    if not files:
        ended_timestamp = datetime.now(timezone.utc)
        status = STATUS_NO_FILES
        pipeline_step_log_upsert(
            spark, step_log_id, pipeline_run_id, step_sequence,
            notebook_folder, notebook_name, status, started_timestamp,
            layer, target_table, 0, 0, ended_timestamp, None
        )
        dbutils.notebook.exit("No CSV files found at " + SOURCE_PATH)

    bad_files = []
    for file_path in files:
        actual = (
            spark.read.format("csv")
                .option("header", "true")
                .load(file_path)
                .limit(0)
                .columns
        )
        if actual != EXPECTED_SOURCE_COLS:
            bad_files.append((file_path, actual))

    if bad_files:
        raise ValueError(
            f"[{TARGET_TABLE}] Header mismatch in {len(bad_files)} file(s).\n"
            f"Expected: {EXPECTED_SOURCE_COLS}\n"
            + "\n".join(f"  {p}\n    actual: {h}" for p, h in bad_files)
        )

    print(f"[{TARGET_TABLE}] {len(files)} file(s) validated")

except dbutils.NotebookExit:
    raise

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, 0, 0, ended_timestamp, error_message
    )
    raise

In [0]:
# =============================================================================
# Cell 5 — Read, shape, and compute row_hash
# All columns remain StringType (no casting at Bronze).
# row_hash is md5 of all source business columns, pipe-delimited.
# =============================================================================

try:
    raw_df = (
        spark.read
            .format("csv")
            .option("header", "true")
            .option("delimiter", ",")
            .schema(read_schema)
            .load(SOURCE_PATH)
            .withColumn("source_file_path", F.col("_metadata.file_path"))
    )

    bronze_df = (
        raw_df
            .select(
                F.col("OnlineRetailer").alias("online_retailer"),
                F.col("SalesMonth").alias("sales_month"),
                F.col("Title").alias("title"),
                F.col("Vintage").alias("vintage"),
                F.col("Variety").alias("variety"),
                F.col("Score").alias("score"),
                F.col("ListPrice").alias("list_price"),
                F.col("Quantity").alias("quantity"),
                F.col("source_file_path"),
            )
            .withColumn(
                "row_hash",
                F.md5(F.concat_ws("|", *[F.col(c) for c in HASH_SOURCE_COLS]))
            )
            .withColumn("inserted_ts", F.lit(started_timestamp).cast("timestamp"))
            .withColumn("run_id",      F.lit(PIPELINE_RUN_ID))
            .withColumn("store_name",  F.lit(STORE_NAME))
            .select(
                "online_retailer", "sales_month", "title", "vintage",
                "variety", "score", "list_price", "quantity",
                "row_hash",
                "inserted_ts", "run_id", "source_file_path", "store_name",
            )
    )

    rows_read = bronze_df.count()
    print(f"[{TARGET_TABLE}] Prepared {rows_read:,} rows for write")

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, 0, ended_timestamp, error_message
    )
    raise

In [0]:
# =============================================================================
# Cell 6 — MERGE write (idempotent on row_hash)
# WHEN NOT MATCHED only — Bronze rows are never updated in place.
# Changed source data arrives as a new hash version alongside the original.
#
# Row count interpretation:
#   rows_read     = total rows in source files this run
#   rows_inserted = net-new rows added to Bronze (hash not previously seen)
#   rows_skipped  = rows already in Bronze (hash already present)
#   rows_written  = rows_inserted (logged to pipeline_step_log)
#
# Zero rows_inserted is valid: all source rows already exist in Bronze.
# =============================================================================

try:
    pre_count = spark.table(TARGET_TABLE).count()

    bronze_df.createOrReplaceTempView("bronze_staging")
    spark.sql(f"""
        MERGE INTO {TARGET_TABLE} AS target
        USING bronze_staging AS source
        ON target.row_hash = source.row_hash
        WHEN NOT MATCHED
            THEN INSERT *
    """)

    post_count    = spark.table(TARGET_TABLE).count()
    rows_inserted = post_count - pre_count
    rows_skipped  = rows_read - rows_inserted
    rows_written  = rows_inserted

    if rows_inserted + rows_skipped != rows_read:
        raise AssertionError(
            f"[{TARGET_TABLE}] Row count inconsistency: "
            f"read {rows_read:,}, inserted {rows_inserted:,}, skipped {rows_skipped:,}. "
            f"Inserted + skipped must equal read."
        )

    print(
        f"[{TARGET_TABLE}] "
        f"rows_read={rows_read:,}  "
        f"rows_inserted={rows_inserted:,}  "
        f"rows_skipped={rows_skipped:,}"
    )

    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_SUCCEEDED

    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )

    df_ingested_files = bronze_df.select("source_file_path").distinct()

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, 0, ended_timestamp, error_message
    )
    raise


# Ingestion log runs in a separate try/except so a logging failure does not
# roll back a successful Bronze write.
try:
    ingestion_log_insert(
        spark,
        df_ingested_files,
        pipeline_run_id,
        step_log_id,
        STORE_NAME,
        TARGET_TABLE,
        error_message,
        datetime.now(timezone.utc)
    )

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    print(f"[{TARGET_TABLE}] WARNING: ingestion_log_insert failed — Bronze write succeeded.\n{error_message}")
    raise

In [0]:
# =============================================================================
# Cell 7 — Archive
# Move processed source files to the archive subfolder.
# Runs after a successful write. A failure here does not roll back Bronze.
# =============================================================================

try:
    Utils.move_all_files(
        dbutils,
        source_path    = SOURCE_PATH,
        target_path    = f"{SOURCE_PATH}archive",
        create_target  = True,
        skip_dirs      = True,
        use_date_partition = False
    )

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    print(f"[{SOURCE_PATH}archive] WARNING: move_all_files to archive failed — Bronze write succeeded.\n{error_message}")
    raise

In [0]:
%skip
%sql
SELECT * FROM vinoworld.audit.pipeline_step_log ORDER BY step_log_id DESC LIMIT 10

In [0]:
%skip
%sql
SELECT * FROM vinoworld.audit.ingestion_log ORDER BY ingestion_id DESC LIMIT 10